<a href="https://colab.research.google.com/github/yukjidam/video-translator-transcriber/blob/main/video_translator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install -U faster-whisper ffmpeg-python
!apt -yqq install ffmpeg >/dev/null

import torch, platform
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    import subprocess, re
    name = subprocess.check_output(["nvidia-smi","--query-gpu=name","--format=csv,noheader"]).decode().strip()
    print("GPU:", name)
print("Python:", platform.python_version())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.8/38.8 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 3.9 MB/s eta 0:00:00


CUDA available: True
GPU: Tesla T4
Python: 3.12.12


In [ ]:
from google.colab import files
import os, pathlib
import ffmpeg

up = files.upload()
INPUT_PATH = next(iter(up))
print("Uploaded:", INPUT_PATH, f"({os.path.getsize(INPUT_PATH)/1e6:.1f} MB)")

audio_exts = {'.mp3', '.wav', '.m4a', '.aac', '.flac', '.ogg', '.opus'}
video_exts = {'.mp4', '.mov', '.mkv', '.avi', '.wmv', '.flv', '.webm'}
ext = pathlib.Path(INPUT_PATH).suffix.lower()

if ext in audio_exts:
    TARGET = INPUT_PATH
    print("Using audio directly:", TARGET)
else:
    AUDIO_PATH = "audio.m4a"
    print("Detected video. Extracting audio →", AUDIO_PATH)
    (ffmpeg
      .input(INPUT_PATH)
      .output(AUDIO_PATH, vn=None, acodec="aac", audio_bitrate="128k")
      .overwrite_output()
      .run())
    TARGET = AUDIO_PATH
    print("Using extracted audio:", TARGET)


Saving (Audio) distribution_pathfinder_pf2448100_05462d87-b4b9-11f0-9c62-48df37a703be.m4a to (Audio) distribution_pathfinder_pf2448100_05462d87-b4b9-11f0-9c62-48df37a703be.m4a
Uploaded: (Audio) distribution_pathfinder_pf2448100_05462d87-b4b9-11f0-9c62-48df37a703be.m4a (19.7 MB)
Using audio directly: (Audio) distribution_pathfinder_pf2448100_05462d87-b4b9-11f0-9c62-48df37a703be.m4a


In [ ]:
from faster_whisper import WhisperModel
device = "cuda" if torch.cuda.is_available() else "cpu"
model = WhisperModel("large-v3", device=device, compute_type="float16" if device=="cuda" else "int8")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

vocabulary.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.bin:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

In [ ]:
import re

def ts_srt(t):
    ms = int(t*1000); h, r = divmod(ms, 3600000); m, r = divmod(r, 60000); s, ms = divmod(r, 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

def clean_text(s: str) -> str:
    s = re.sub(r"\s+", " ", s or "").strip()
    s = re.sub(r"\s+([,.!?;:])", r"\1", s)
    return s

segments_en, info_en = model.transcribe(
    TARGET,
    task="translate",
    language=None,
    vad_filter=True,
    condition_on_previous_text=True,
    beam_size=5,
    word_timestamps=False
)

stock = [{"st": s.start, "ed": s.end, "txt": clean_text(s.text)} for s in segments_en if clean_text(s.text)]
out_stock = "subtitles.en.srt"
with open(out_stock, "w", encoding="utf-8") as f:
    for i, seg in enumerate(stock, 1):
        f.write(f"{i}\n{ts_srt(seg['st'])} --> {ts_srt(seg['ed'])}\n{seg['txt']}\n\n")

print("Wrote:", out_stock, "| segments:", len(stock), "| primary:", info_en.language)


Wrote: subtitles.en.srt | segments: 993 | primary: ko


In [ ]:
import os, re, glob, subprocess, json, wave
from pathlib import Path
import numpy as np

INPUT_SRT = "subtitles.en.srt"
GAP_SEC   = 0.04
MIN_DUR   = 0.25


VAD_MIN_SIL_MS    = 900
VAD_MIN_SPEECH_MS = 180
MERGE_GAP_SEC     = 0.06


def ts_to_sec(ts: str) -> float:
    h, m, rest = ts.split(":")
    s, ms = rest.split(",")
    return int(h)*3600 + int(m)*60 + int(s) + int(ms)/1000.0

def sec_to_ts(t: float) -> str:
    if t < 0: t = 0.0
    ms = int(round(t*1000))
    h, r = divmod(ms, 3600000)
    m, r = divmod(r, 60000)
    s, ms = divmod(r, 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

def parse_srt(path: str):
    blocks, cur = [], []
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            if line.strip():
                cur.append(line.rstrip("\n"))
            else:
                if cur:
                    blocks.append(cur); cur=[]
        if cur: blocks.append(cur)
    entries = []
    for b in blocks:
        if len(b) < 2: continue
        m = re.search(r"(\d{2}:\d{2}:\d{2},\d{3})\s*-->\s*(\d{2}:\d{2}:\d{2},\d{3})", b[1])
        if not m: continue
        st, ed = ts_to_sec(m.group(1)), ts_to_sec(m.group(2))
        txt = "\n".join(b[2:]).strip()
        entries.append({"st": st, "ed": ed, "txt": txt})
    return entries

def write_srt(entries, path):
    with open(path, "w", encoding="utf-8") as f:
        for i, e in enumerate(entries, 1):
            f.write(f"{i}\n{sec_to_ts(e['st'])} --> {sec_to_ts(e['ed'])}\n{e['txt']}\n\n")

def clean_space(s: str) -> str:
    s = re.sub(r"\s+", " ", s or "").strip()
    s = re.sub(r"\s+([,.!?;:])", r"\1", s)
    return s

def sentence_split(text: str):
    text = clean_space(text or "")
    if not text: return []
    tokens = re.split(r'([.!?…]["\']?)', text)
    out = []
    i = 0
    while i < len(tokens):
        chunk = tokens[i].strip()
        punct = tokens[i+1] if i+1 < len(tokens) else ''
        if chunk:
            out.append(clean_space(chunk + (punct or "")))
        i += 2
    return out if out else [text]

def split_text_to_n_chunks(text: str, n: int, durations=None):
    text = clean_space(text)
    if n <= 1: return [text]
    sents = sentence_split(text)
    if len(sents) == n:
        return sents
    if len(sents) > n:
        return sents[:n-1] + [" ".join(sents[n-1:])]
    words = text.split()
    total = len(words)
    if total == 0: return [""]*n
    if durations and sum(durations) > 0:
        props = [d/sum(durations) for d in durations]
    else:
        props = [1.0/n]*n
    counts = [max(1, int(round(p*total))) for p in props]
    diff = sum(counts) - total
    k = 0
    while diff != 0 and k < 1000:
        for i in range(n):
            if diff > 0 and counts[i] > 1:
                counts[i] -= 1; diff -= 1
                if diff == 0: break
            elif diff < 0:
                counts[i] += 1; diff += 1
                if diff == 0: break
        k += 1
    chunks, idx = [], 0
    for c in counts:
        chunks.append(" ".join(words[idx: idx+c])); idx += c
    return [clean_space(c) for c in chunks]

def intersect(a0,a1,b0,b1):
    lo, hi = max(a0,b0), min(a1,b1)
    return (lo,hi) if hi > lo else None

def _has_ffmpeg():
    try:
        subprocess.run(["ffmpeg","-version"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
        return True
    except Exception:
        return False
if "TARGET" not in globals():
    raise RuntimeError("TARGET not found. Make sure you ran c2 (upload) so TARGET is set.")
if not os.path.exists(INPUT_SRT):
    srts = sorted(glob.glob("*.srt"), key=os.path.getmtime)
    if not srts:
        raise FileNotFoundError("No .srt found. Run c4a first to create subtitles.en.srt")
    INPUT_SRT = srts[-1]
print("Audio:", TARGET)
print("SRT  :", INPUT_SRT)

tmp_wav = "_vad_tmp_16k.wav"
if not _has_ffmpeg():
    raise RuntimeError("ffmpeg not found. Ensure c1 ran successfully.")
subprocess.run(
    ["ffmpeg","-y","-i",TARGET,"-ac","1","-ar","16000","-f","wav",tmp_wav],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True
)

def read_wav_mono16(path):
    with wave.open(path, "rb") as w:
        n_channels = w.getnchannels()
        sampwidth  = w.getsampwidth()
        framerate  = w.getframerate()
        n_frames   = w.getnframes()
        raw = w.readframes(n_frames)
    assert sampwidth == 2, "Expected 16-bit PCM WAV"
    data = np.frombuffer(raw, dtype=np.int16)
    if n_channels == 2:
        data = data.reshape(-1,2).mean(axis=1).astype(np.int16)
    audio = data.astype(np.float32) / 32768.0
    return audio, framerate

audio, sr = read_wav_mono16(tmp_wav)
duration = len(audio)/sr
print(f"WAV ok: sr={sr}, dur={duration:.2f}s")

frame_ms = 20
hop_ms   = 10
frame_len = int(sr*frame_ms/1000)
hop_len   = int(sr*hop_ms/1000)

def frame_rms(x, frame_len, hop_len):
    n = len(x)
    if n < frame_len:
        return np.array([np.sqrt(np.mean(x**2) + 1e-12)], dtype=np.float32)
    out = []
    for i in range(0, n - frame_len + 1, hop_len):
        seg = x[i:i+frame_len]
        out.append(np.sqrt(np.mean(seg**2) + 1e-12))
    return np.array(out, dtype=np.float32)

rms = frame_rms(audio, frame_len, hop_len)
win = max(1, int(100/hop_ms))
kernel = np.ones(win)/win
rms_s = np.convolve(rms, kernel, mode='same')

med = np.median(rms_s)
p95 = np.percentile(rms_s, 95)
thr = max(1e-3, med + 0.35*(p95 - med), med*2.0)
mask = rms_s > thr

min_sil_frames    = max(1, int(VAD_MIN_SIL_MS / hop_ms))
min_speech_frames = max(1, int(VAD_MIN_SPEECH_MS / hop_ms))

def close_small_gaps(mask, max_gap):
    m = mask.copy()
    i=0; L=len(m)
    while i<L:
        if not m[i]:
            j=i
            while j<L and not m[j]: j+=1
            gap = j-i
            if 0 < gap <= max_gap:
                m[i:j] = True
            i=j
        else:
            i+=1
    return m

def remove_short_bursts(mask, min_len):
    m = mask.copy()
    i=0; L=len(m)
    while i<L:
        if m[i]:
            j=i
            while j<L and m[j]: j+=1
            if (j-i) < min_len:
                m[i:j] = False
            i=j
        else:
            i+=1
    return m

mask = close_small_gaps(mask, max_gap=min_sil_frames-1)
mask = remove_short_bursts(mask, min_len=min_speech_frames)

intervals = []
i=0; L=len(mask)
while i<L:
    if mask[i]:
        j=i
        while j<L and mask[j]: j+=1
        st = (i*hop_len)/sr
        ed = min(duration, (j*hop_len + frame_len)/sr)
        if ed - st > 0.0:
            intervals.append((st, ed))
        i=j
    else:
        i+=1
merged = []
for (st,ed) in intervals:
    if not merged:
        merged.append([st,ed]); continue
    pst,ped = merged[-1]
    if st - ped <= MERGE_GAP_SEC:
        merged[-1][1] = max(ped, ed)
    else:
        merged.append([st,ed])
intervals = [tuple(x) for x in merged]
print(f"Speech intervals: {len(intervals)}")

subs = parse_srt(INPUT_SRT)

retimed = []
for s in subs:
    overlaps = []
    for (a0,a1) in intervals:
        lo, hi = max(s["st"], a0), min(s["ed"], a1)
        if hi > lo:
            overlaps.append((lo, hi))
    if not overlaps:
        continue

    if len(overlaps) == 1:
        st, ed = overlaps[0]
        if ed - st < MIN_DUR: ed = min(duration, st + MIN_DUR)
        retimed.append({"st": st, "ed": ed, "txt": s["txt"]})
    else:
        durs = [b - a for (a, b) in overlaps]
        parts = split_text_to_n_chunks(s["txt"], len(overlaps), durations=durs)
        for (st, ed), txt in zip(overlaps, parts):
            if not txt.strip(): continue
            if ed - st < MIN_DUR: ed = min(duration, st + MIN_DUR)
            retimed.append({"st": st, "ed": ed, "txt": txt})

retimed.sort(key=lambda e: (e["st"], e["ed"]))
for e in retimed:
    if (e["ed"] - e["st"]) < MIN_DUR:
        e["ed"] = min(duration, e["st"] + MIN_DUR)

for i in range(len(retimed) - 1):
    a, b = retimed[i], retimed[i+1]
    if a["ed"] >= b["st"]:
        a["ed"] = max(a["st"] + 0.01, b["st"] - GAP_SEC)
        if b["st"] <= a["ed"]:
            b["st"] = min(duration, a["ed"] + GAP_SEC)

OUT_SRT = f"{os.path.splitext(INPUT_SRT)[0]}.retimed.srt"
write_srt(retimed, OUT_SRT)

print(json.dumps({
    "out_path": OUT_SRT,
    "subs_in": len(subs),
    "subs_out": len(retimed),
    "intervals": len(intervals)
}, indent=2))


Audio: (Audio) distribution_pathfinder_pf2448100_05462d87-b4b9-11f0-9c62-48df37a703be.m4a
SRT  : subtitles.en.srt
WAV ok: sr=16000, dur=4588.63s
Speech intervals: 744
{
  "out_path": "subtitles.en.retimed.srt",
  "subs_in": 993,
  "subs_out": 873,
  "intervals": 744
}


In [ ]:
import os, glob
from google.colab import files

def download_srt(name_contains=None):
    candidates = sorted(glob.glob("*.srt"), key=os.path.getmtime)
    if not candidates:
        raise FileNotFoundError("No .srt files found.")
    if name_contains:
        matches = [p for p in candidates if name_contains.lower() in os.path.basename(p).lower()]
        if not matches:
            raise FileNotFoundError(f'No .srt matching "{name_contains}". Available: {candidates}')
        target = matches[-1]
    else:
        target = candidates[-1]
    print("Downloading:", target)
    files.download(target)


download_srt()

SyntaxError: invalid syntax (ipython-input-2910199825.py, line 19)